In [25]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/ahmedtalima45/my-data/train_fixed.xlsx
/kaggle/input/datasets/ahmedtalima45/my-data/validation_fixed.xlsx
/kaggle/input/datasets/ahmedtalima45/my-data/unlabeled_fixed.xlsx


In [26]:
import pandas as pd
import ast

def load_data(path):
    df = pd.read_excel(path)

    # Convert columns if they exist
    if "aspects" in df.columns:
        df["aspects"] = df["aspects"].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else x
        )

    if "aspect_sentiments" in df.columns:
        df["aspect_sentiments"] = df["aspect_sentiments"].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else x
        )

    return df


# Load datasets
train_df = load_data("/kaggle/input/datasets/ahmedtalima45/my-data/train_fixed.xlsx")
val_df = load_data("/kaggle/input/datasets/ahmedtalima45/my-data/validation_fixed.xlsx")
test_df = load_data("/kaggle/input/datasets/ahmedtalima45/my-data/unlabeled_fixed.xlsx")

# Basic checks
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain sample:")
print(train_df.iloc[0])

print("\nTest sample:")
print(test_df.iloc[0])

Train: (1971, 9)
Validation: (500, 9)
Test: (7047, 7)

Train sample:
review_id                                                         7238
review_text                        لا يوجد الدفع بالبطاقه عند الاستلام
star_rating                                                          3
date                                               2026-03-08 00:00:00
business_name                                                     Noon
business_category                                            ecommerce
platform                                                    play_store
aspects                                     [app_experience, delivery]
aspect_sentiments    {'app_experience': 'negative', 'delivery': 'ne...
Name: 0, dtype: object

Test sample:
review_id                                                            1
review_text          Incroyablement grand avec des belles boutiques...
star_rating                                                          5
date                                      

In [27]:
ASPECTS = [
    "food", "service", "price", "cleanliness",
    "delivery", "ambiance", "app_experience",
    "general", "none"
]

In [28]:
def encode_aspects(aspect_list):
    return [1 if aspect in aspect_list else 0 for aspect in ASPECTS]

In [29]:
print("Train columns:", train_df.columns)
print("Validation columns:", val_df.columns)

Train columns: Index(['review_id', 'review_text', 'star_rating', 'date', 'business_name',
       'business_category', 'platform', 'aspects', 'aspect_sentiments'],
      dtype='object')
Validation columns: Index(['review_id', 'review_text', 'star_rating', 'date', 'business_name',
       'business_category', 'platform', 'aspects', 'aspect_sentiments'],
      dtype='object')


In [30]:
train_df["aspect_labels"] = train_df["aspects"].apply(encode_aspects)
val_df["aspect_labels"] = val_df["aspects"].apply(encode_aspects)

In [31]:
for i in range(3):
    print("Text:", train_df.loc[i, "review_text"])
    print("Aspects:", train_df.loc[i, "aspects"])
    print("Encoded:", train_df.loc[i, "aspect_labels"])
    print("-" * 50)

Text: لا يوجد الدفع بالبطاقه عند الاستلام
Aspects: ['app_experience', 'delivery']
Encoded: [0, 0, 0, 0, 1, 0, 1, 0, 0]
--------------------------------------------------
Text: المكان نضيف وجميل وقعدته تحفه والخدمة فوق الممتاز والجو جميل مكان اكتر من رائع بصراحة ❤️❤️❤️❤️
Aspects: ['cleanliness', 'ambiance', 'service']
Encoded: [0, 1, 0, 1, 0, 1, 0, 0, 0]
--------------------------------------------------
Text: تجربة سيئة سألتهم الاكل هياخد وقت قد ايه قالولي نص ساعة فعد ساعة ونص
الاكل يعني كويس ولكن كمية قليلة جدا
Aspects: ['service', 'delivery', 'food']
Encoded: [1, 1, 0, 0, 1, 0, 0, 0, 0]
--------------------------------------------------


In [32]:
print(train_df[train_df["aspects"].apply(lambda x: "none" in x)].head())

     review_id                                        review_text  \
5         8003  لقد حملت تطبيق عن طريق مستر بيست واريد سيارة ت...   
151       7673                                              شكران   
171       8016  انا حملت هذا التطبيق من مستر بيست واتمني أن يح...   
173        115                                              Shady   
227       4995                                         لوكيشن خطأ   

     star_rating                 date business_name business_category  \
5              1  2024-08-17 00:00:00          Shop         ecommerce   
151            5  2026-03-12 00:00:00        Amazon         ecommerce   
171            5  2024-04-14 00:00:00          Shop         ecommerce   
173            5          قبل 11 ساعة     شتيجنبرجر              فندق   
227            1            قبل سنتين        كازارى  متجر ملابس رجالي   

        platform aspects    aspect_sentiments                aspect_labels  
5     play_store  [none]  {'none': 'neutral'}  [0, 0, 0, 0, 0, 0, 0, 

In [33]:
import re

# -------------------------
# Emoji mapping (important for sentiment)
# -------------------------
emoji_map = {
    "❤️": " حب ",
    "😍": " حب ",
    "😂": " ضحك ",
    "😡": " غضب ",
    "👍": " جيد ",
    "👎": " سيء "
}

# -------------------------
# Remove elongation (هههههه → هه)
# -------------------------
def remove_elongation(text):
    return re.sub(r"(.)\1{2,}", r"\1\1", text)

# -------------------------
# Normalize Arabic
# -------------------------
def normalize_arabic(text):
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "و", text)
    text = re.sub("ئ", "ي", text)
    text = re.sub("ة", "ه", text)
    return text

# -------------------------
# Remove diacritics
# -------------------------
def remove_diacritics(text):
    arabic_diacritics = re.compile("""
        ّ|َ|ً|ُ|ٌ|ِ|ٍ|ْ|ـ
    """, re.VERBOSE)
    return re.sub(arabic_diacritics, '', text)

# -------------------------
# Replace emojis with meaning
# -------------------------
def replace_emojis(text):
    for emoji, word in emoji_map.items():
        text = text.replace(emoji, word)
    return text

# -------------------------
# Remove URLs and noise
# -------------------------
def remove_noise(text):
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# -------------------------
# FULL PIPELINE
# -------------------------
def preprocess_text(text):
    text = str(text)

    text = text.lower()
    text = replace_emojis(text)
    text = remove_elongation(text)
    text = normalize_arabic(text)
    text = remove_diacritics(text)
    text = remove_noise(text)

    return text

In [34]:
aspect_train = train_df[["review_text", "aspect_labels"]]
aspect_val = val_df[["review_text", "aspect_labels"]]

In [35]:
aspect_train["review_text"] = aspect_train["review_text"].apply(preprocess_text)
aspect_val["review_text"] = aspect_val["review_text"].apply(preprocess_text)

/tmp/ipykernel_55/897094203.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  aspect_train["review_text"] = aspect_train["review_text"].apply(preprocess_text)
/tmp/ipykernel_55/897094203.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  aspect_val["review_text"] = aspect_val["review_text"].apply(preprocess_text)


In [36]:
print(len(ASPECTS))  # should be 9
print(len(train_df.loc[0, "aspect_labels"]))  # should also be 9

9
9


In [37]:
import pandas as pd

def build_sentiment_dataset(df):
    rows = []

    for _, row in df.iterrows():
        text = row["review_text"]
        aspects = row["aspects"]
        sentiments = row["aspect_sentiments"]

        for aspect in aspects:
            rows.append({
                "text": text,
                "aspect": aspect,
                "sentiment": sentiments[aspect]
            })

    return pd.DataFrame(rows)

In [38]:
train_sentiment_df = build_sentiment_dataset(train_df)
val_sentiment_df = build_sentiment_dataset(val_df)

print("Train sentiment size:", len(train_sentiment_df))
print("Val sentiment size:", len(val_sentiment_df))

Train sentiment size: 3333
Val sentiment size: 840


In [39]:
label_map = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

train_sentiment_df["label"] = train_sentiment_df["sentiment"].map(label_map)
val_sentiment_df["label"] = val_sentiment_df["sentiment"].map(label_map)

In [40]:
def build_input(text, aspect):
    return f"[ASPECT] {aspect} [TEXT] {text}"

train_sentiment_df["input_text"] = train_sentiment_df.apply(
    lambda x: build_input(x["text"], x["aspect"]),
    axis=1
)

val_sentiment_df["input_text"] = val_sentiment_df.apply(
    lambda x: build_input(x["text"], x["aspect"]),
    axis=1
)

In [41]:
print(train_sentiment_df.head())
print(train_sentiment_df["label"].value_counts())

print(val_sentiment_df.head())

                                                text          aspect  \
0                لا يوجد الدفع بالبطاقه عند الاستلام  app_experience   
1                لا يوجد الدفع بالبطاقه عند الاستلام        delivery   
2  المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...     cleanliness   
3  المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...        ambiance   
4  المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...         service   

  sentiment  label                                         input_text  
0  negative      0  [ASPECT] app_experience [TEXT] لا يوجد الدفع ب...  
1  negative      0  [ASPECT] delivery [TEXT] لا يوجد الدفع بالبطاق...  
2  positive      2  [ASPECT] cleanliness [TEXT] المكان نضيف وجميل ...  
3  positive      2  [ASPECT] ambiance [TEXT] المكان نضيف وجميل وقع...  
4  positive      2  [ASPECT] service [TEXT] المكان نضيف وجميل وقعد...  
label
2    1646
0    1538
1     149
Name: count, dtype: int64
                                                text          aspect  \
0

In [42]:
train_sentiment_df.head()

,text,aspect,sentiment,label,input_text
0,لا يوجد الدفع بالبطاقه عند الاستلام,app_experience,negative,0,[ASPECT] app_experience [TEXT] لا يوجد الدفع ب...
1,لا يوجد الدفع بالبطاقه عند الاستلام,delivery,negative,0,[ASPECT] delivery [TEXT] لا يوجد الدفع بالبطاق...
2,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,cleanliness,positive,2,[ASPECT] cleanliness [TEXT] المكان نضيف وجميل ...
3,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,ambiance,positive,2,[ASPECT] ambiance [TEXT] المكان نضيف وجميل وقع...
4,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,service,positive,2,[ASPECT] service [TEXT] المكان نضيف وجميل وقعد...


In [43]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

In [44]:
MODEL_NAME = "aubmindlab/bert-base-arabertv2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/384 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/611 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [45]:
class AspectDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(label, dtype=torch.float)
        }

In [46]:
train_texts = train_df["review_text"].values
train_labels = train_df["aspect_labels"].values

val_texts = val_df["review_text"].values
val_labels = val_df["aspect_labels"].values

In [47]:
train_dataset = AspectDataset(train_texts, train_labels, tokenizer)
val_dataset = AspectDataset(val_texts, val_labels, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)

In [48]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(ASPECTS),
    problem_type="multi_label_classification"
)

model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: aubmindlab/bert-base-arabertv2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider

In [49]:
loss_fn = torch.nn.BCEWithLogitsLoss()

In [50]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5)

In [ ]:
epochs = 3

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["labels"]

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = loss_fn(outputs.logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

In [ ]:
from sklearn.metrics import f1_score
import numpy as np
import torch

def evaluate_aspect_model(model, dataloader, threshold=0.5, device="cpu"):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:

            input_ids = batch["input_ids"]
            attention_mask = batch["attention_mask"]
            labels = batch["labels"].cpu().numpy()

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

            probs = torch.sigmoid(outputs.logits).cpu().numpy()
            preds = (probs > threshold).astype(int)

            all_preds.append(preds)
            all_labels.append(labels)

    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)

    f1 = f1_score(all_labels, all_preds, average="macro")

    print("Macro F1:", f1)

    return f1

In [ ]:
val_f1 = evaluate_aspect_model(model, val_loader)

In [ ]:
threshold = 0.5

In [ ]:
best_t = 0.5
best_f1 = 0

for t in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:

    f1 = evaluate_aspect_model(model, val_loader, threshold=t)

    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print("Best threshold:", best_t)
print("Best F1:", best_f1)

In [ ]:
save_path = "aspect_model"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained("aspect_model")
tokenizer = AutoTokenizer.from_pretrained("aspect_model")

In [ ]:
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
}, "aspect_training_checkpoint.pth")

In [ ]:
checkpoint = torch.load("aspect_training_checkpoint.pth")

model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

In [ ]:
import json

with open("aspect_list.json", "w") as f:
    json.dump(ASPECTS, f)

In [ ]:
import re

# -------------------------
# Emoji mapping (important for sentiment)
# -------------------------
emoji_map = {
    "❤️": " حب ",
    "😍": " حب ",
    "😂": " ضحك ",
    "😡": " غضب ",
    "👍": " جيد ",
    "👎": " سيء "
}

# -------------------------
# Remove elongation (هههههه → هه)
# -------------------------
def remove_elongation(text):
    return re.sub(r"(.)\1{2,}", r"\1\1", text)

# -------------------------
# Normalize Arabic
# -------------------------
def normalize_arabic(text):
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    text = re.sub("ؤ", "و", text)
    text = re.sub("ئ", "ي", text)
    text = re.sub("ة", "ه", text)
    return text

# -------------------------
# Remove diacritics
# -------------------------
def remove_diacritics(text):
    arabic_diacritics = re.compile("""
        ّ|َ|ً|ُ|ٌ|ِ|ٍ|ْ|ـ
    """, re.VERBOSE)
    return re.sub(arabic_diacritics, '', text)

# -------------------------
# Replace emojis with meaning
# -------------------------
def replace_emojis(text):
    for emoji, word in emoji_map.items():
        text = text.replace(emoji, word)
    return text

# -------------------------
# Remove URLs and noise
# -------------------------
def remove_noise(text):
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# -------------------------
# FULL PIPELINE
# -------------------------
def preprocess_text(text):
    text = str(text)

    text = text.lower()
    text = replace_emojis(text)
    text = remove_elongation(text)
    text = normalize_arabic(text)
    text = remove_diacritics(text)
    text = remove_noise(text)

    return text